### 02 - Notebook para limpieza y preparación de datos

Idea de limpieza:
- Abrir los 3 CSVs en dfs
- Estandarizar columnas: Añadir columnas al df3 (hosp_uci_def_sexo_edad_provres_todas_edades.csv)
- Concatenar los 3 dfs en un solo df
- Limpiar formatos 
- Eliminar posibles duplicados
- Ordenar cronologicamente
- Guardar Tabla Maestra

In [10]:
# Importar librerias necesarias
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

# Leer los datos
f1 = RAW_DATA_DIR / "casos_hosp_uci_def_sexo_edad_provres.csv"
f2 = RAW_DATA_DIR / "casos_hosp_uci_def_sexo_edad_provres_60_mas.csv"
f3 = RAW_DATA_DIR / "hosp_uci_def_sexo_edad_provres_todas_edades.csv"

df1 = pd.read_csv(f1)
df2 = pd.read_csv(f2)
df3 = pd.read_csv(f3)

Archivos abiertos, procedemos a toda la limpieza y estructuración de los datos

In [11]:
# Estandarizar columnas num_casos en el df3 ( que no tiene ) con valor 0 o NaN
if "num_casos" in df3.columns:
    df3["num_casos"] = 0
# Concatenar los datos
master_df = pd.concat([df1,df2,df3], axis=0, ignore_index=True)
# Limpieza de formato
master_df["fecha"] = pd.to_datetime(master_df["fecha"])

# Eliminar posibles duplicados ya que estamos concatenando 3 tablas
master_df = master_df.drop_duplicates(subset=["fecha", "provincia_iso", "sexo", "grupo_edad"])

# ordenar cronologicamente
master_df = master_df.sort_values(by=["fecha", "provincia_iso"])

# Sustituir NaN por 0
master_df["num_casos"] = master_df["num_casos"].fillna(0).astype(int)

* Hecho toda la limpieza, se procese a guardar la tabla maestra en la nueva ruta de carpeta procesada

In [14]:
# Guardar master_df
master_df.to_csv(PROCESSED_DATA_DIR / "master_df.csv", index=False)
print(f"✅ Tabla maestra creada con {master_df.shape[0]} filas")

# Guardo tambien un resumen de la tabla
df_daily = master_df.groupby(["fecha", "provincia_iso"])[["num_casos", "num_hosp", "num_uci", "num_def"]].sum().reset_index()
df_daily.to_csv(PROCESSED_DATA_DIR / "daily_stats_spain.csv", index= False)

✅ Tabla maestra creada con 2038380 filas


In [13]:
# Verificar si hay salto temporal
rango_esperado = pd.date_range(start=master_df["fecha"].min(), end=master_df["fecha"].max())
dias_faltantes = rango_esperado.difference(master_df["fecha"].unique())
if dias_faltantes.empty:
    print("✅ No hay dias faltantes en el dataset.")
else:
    print(f"⚠️ Hay {len(dias_faltantes)} dias faltantes en el dataset.")

✅ No hay dias faltantes en el dataset.
